In [5]:
import time
from adafruit_extended_bus import ExtendedI2C
from adafruit_servokit import ServoKit

I2C_BUS_CANDIDATES = tuple(range(0, 17))


def find_pca9685_controller():
    last_error = None
    for bus in I2C_BUS_CANDIDATES:
        try:
            i2c = ExtendedI2C(bus)
            kit = ServoKit(channels=16, i2c=i2c)
            servo_ch1 = kit.servo[1]
            servo_ch2 = kit.servo[2]
            return bus, i2c, kit, servo_ch1, servo_ch2
        except Exception as exc:
            last_error = exc
            continue
    raise RuntimeError(
        f"No PCA9685 controller detected on any tested I2C bus {I2C_BUS_CANDIDATES}. "
        f"Check the CH341T wiring and the active bus number. Last error: {last_error}"
    )


def main():
    servo_ch1 = None
    servo_ch2 = None
    i2c = None
    bus = None

    print("Scanning I2C buses for a PCA9685 controller...")
    try:
        bus, i2c, kit, servo_ch1, servo_ch2 = find_pca9685_controller()
        print(f"Detected PCA9685 on I2C bus {bus}.")

        print("Starting servo loop. Interrupt the kernel to stop.")
        while True:
            print("Moving to 0 degrees...")
            servo_ch1.angle = 0
            servo_ch2.angle = 0
            time.sleep(1.5)

            print("Moving to 90 degrees...")
            servo_ch1.angle = 90
            servo_ch2.angle = 90
            time.sleep(1.5)

            print("Moving to 180 degrees...")
            servo_ch1.angle = 180
            servo_ch2.angle = 180
            time.sleep(1.5)

            print("Performing gradual sweep back...")
            for angle in range(180, -1, -5):
                servo_ch1.angle = angle
                servo_ch2.angle = 180 - angle
                time.sleep(0.05)
    except KeyboardInterrupt:
        print("\nTest stopped by user. Relaxing servos.")
    except Exception as exc:
        print(f"\nServo test failed on bus {bus}: {exc}")
        raise
    finally:
        for servo in (servo_ch1, servo_ch2):
            if servo is not None:
                try:
                    servo.angle = None
                except Exception as cleanup_error:
                    print(f"Could not relax servo: {cleanup_error}")
        if i2c is not None:
            try:
                i2c.deinit()
            except Exception:
                pass


if __name__ == "__main__":
    main()

Scanning I2C buses for a PCA9685 controller...

Servo test failed on bus None: No PCA9685 controller detected on any tested I2C bus (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16). Check the CH341T wiring and the active bus number. Last error: No device found for /dev/i2c-16


RuntimeError: No PCA9685 controller detected on any tested I2C bus (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16). Check the CH341T wiring and the active bus number. Last error: No device found for /dev/i2c-16